# Homework 11: Evaluation and Risk Communication

Bootstrap confidence intervals (gaussian-based vs bootstrap-based), a three-way scenario
comparison on missing-data handling, and a subgroup diagnostic by segment, on a synthetic
dataset with heavy-tailed noise and 5% missingness built in on purpose.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.evaluation import (
    mean_impute, median_impute, SimpleLinReg, mae,
    fit_fn, pred_fn, bootstrap_metric, bootstrap_predictions,
)

np.random.seed(111)
plt.rcParams['figure.figsize'] = (8, 5)

## Load Data
Generated once and cached to `data/raw/data_stage11_eval_risk.csv`.

In [2]:
raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)
csv_path = raw_dir / 'data_stage11_eval_risk.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path, parse_dates=['date'])
else:
    n = 180
    dates = pd.date_range('2022-06-01', periods=n, freq='D')
    seg = np.random.choice(['A', 'B', 'C'], size=n, p=[0.5, 0.3, 0.2])
    x = np.linspace(0, 9, n) + np.random.normal(0, 0.7, n)
    y = 2.1 * x + 0.8 + np.random.standard_t(df=3, size=n) * 1.1
    x[np.random.choice(np.arange(n), size=round(0.05 * n), replace=False)] = np.nan
    df = pd.DataFrame({'date': dates, 'segment': seg, 'x_feature': x, 'y_target': y})
    df.to_csv(csv_path, index=False)

df.head()

,date,segment,x_feature,y_target
0,2022-06-01,B,0.547868,2.107524
1,2022-06-02,A,0.974480,2.209111
2,2022-06-03,A,-0.012991,0.867315
3,2022-06-04,B,-1.012503,-1.741932
4,2022-06-05,A,0.642399,1.615007


## Baseline Fit
Mean imputation for missing `x_feature`, then a simple linear fit.

In [3]:
X_raw = df['x_feature'].values
y = df['y_target'].values
X_base = mean_impute(X_raw)

model = fit_fn(X_base.reshape(-1, 1), y)
y_hat = pred_fn(model, X_base.reshape(-1, 1))
df['x_imputed'] = X_base

base_mae = mae(y, y_hat)
print(f'Baseline MAE (mean imputation): {base_mae:.4f}')
print(f'Missing values in x_feature: {np.isnan(X_raw).sum()} of {len(X_raw)}')

Baseline MAE (mean imputation): 1.2783
Missing values in x_feature: 9 of 180


## Gaussian-Based vs Bootstrap-Based CI

The gaussian band assumes residuals are normal and uses `sigma_hat / sqrt(n)` for the standard
error of the mean prediction. The bootstrap band makes no such assumption: it refits on 600
resamples and takes the 2.5th/97.5th percentile of the resulting prediction lines directly.
Since `y` was generated with `t`-distributed (3 df) noise, not gaussian noise, this is a direct
test of whether that assumption actually matters here.

In [4]:
resid = y - y_hat
sigma_hat = np.std(resid, ddof=1)
n = len(y)
se_mean = sigma_hat / np.sqrt(n)

x_grid = np.linspace(np.nanmin(X_base), np.nanmax(X_base), 120).reshape(-1, 1)
pred_line = pred_fn(model, x_grid)
gauss_lo = pred_line - 1.96 * se_mean
gauss_hi = pred_line + 1.96 * se_mean

m_boot, lo_boot, hi_boot = bootstrap_predictions(X_base, y, x_grid, n_boot=600)

fig, ax = plt.subplots()
ax.scatter(X_base, y, alpha=0.25)
ax.plot(x_grid, pred_line, label='prediction', color='black')
ax.fill_between(x_grid.ravel(), gauss_lo, gauss_hi, alpha=0.3, label='gaussian-based CI')
ax.fill_between(x_grid.ravel(), lo_boot, hi_boot, alpha=0.3, label='bootstrap CI')
ax.legend()
ax.set_title('Gaussian-based vs bootstrap-based CI')
fig.tight_layout()
fig.savefig('data/processed/ci_comparison.png', dpi=110)
plt.close(fig)

gauss_width = float(np.mean(gauss_hi - gauss_lo))
boot_width = float(np.mean(hi_boot - lo_boot))
print(f'Mean gaussian CI width: {gauss_width:.4f}')
print(f'Mean bootstrap CI width: {boot_width:.4f}')
print('Saved data/processed/ci_comparison.png')

Mean gaussian CI width: 0.6100
Mean bootstrap CI width: 0.7909
Saved data/processed/ci_comparison.png


## Scenario Sensitivity: Missing-Data Handling

Three scenarios for the 5% missing `x_feature` values: mean imputation, median imputation, and
dropping the missing rows outright.

In [5]:
scenarios = {
    'mean_impute': mean_impute,
    'median_impute': median_impute,
    'drop_missing': lambda a: a[~np.isnan(a)] if np.isnan(a).any() else a,
}

results = []
for name, fn in scenarios.items():
    if name == 'drop_missing' and np.isnan(X_raw).any():
        mask = ~np.isnan(X_raw)
        Xs, ys = X_raw[mask], y[mask]
    else:
        Xs, ys = fn(X_raw), y
    m = fit_fn(Xs.reshape(-1, 1), ys)
    yh = pred_fn(m, Xs.reshape(-1, 1))
    results.append({'scenario': name, 'mae': mae(ys, yh), 'slope': m.coef_[0], 'intercept': m.intercept_, 'n': len(ys)})

sens = pd.DataFrame(results)
Path('data/processed').mkdir(parents=True, exist_ok=True)
sens.to_csv('data/processed/scenario_results.csv', index=False)
sens

,scenario,mae,slope,intercept,n
0,mean_impute,1.278317,2.130236,0.711523,180
1,median_impute,1.283954,2.129290,0.727146,180
2,drop_missing,1.064603,2.130236,0.659164,171


In [6]:
fig, ax = plt.subplots()
xg = np.linspace(np.nanmin(X_base), np.nanmax(X_base), 150).reshape(-1, 1)
for name, fn in scenarios.items():
    if name == 'drop_missing' and np.isnan(X_raw).any():
        mask = ~np.isnan(X_raw)
        Xi, yi = X_raw[mask], y[mask]
    else:
        Xi, yi = fn(X_raw), y
    m = fit_fn(Xi.reshape(-1, 1), yi)
    ax.plot(xg, pred_fn(m, xg), label=name)
ax.scatter(X_base, y, alpha=0.15)
ax.set_title('Scenario fits (consistent axes)')
ax.legend()
fig.tight_layout()
fig.savefig('data/processed/scenario_fits.png', dpi=110)
plt.close(fig)
print('Saved data/processed/scenario_fits.png')

Saved data/processed/scenario_fits.png


## Subgroup Diagnostic: by Segment

In [7]:
model_base = fit_fn(X_base.reshape(-1, 1), y)
df2 = df.copy()
df2['y_hat'] = pred_fn(model_base, df2['x_imputed'].values.reshape(-1, 1))
df2['resid'] = df2['y_target'] - df2['y_hat']

subgroup = df2.groupby('segment')['resid'].agg(['mean', 'std', 'median', 'count'])
subgroup

,mean,std,median,count
segment,,,,
A,-0.066306,1.827501,-0.235648,94
B,0.294967,1.707514,-0.077390,41
C,-0.130242,2.813215,-0.140354,45


In [8]:
fig, ax = plt.subplots()
grouped = df2.groupby('segment')['resid']
data = [s.values for _, s in grouped]
labels = list(grouped.groups.keys())
ax.boxplot(data)
ax.set_xticks(range(1, len(labels) + 1))
ax.set_xticklabels(labels)
ax.set_title('Residuals by segment')
fig.tight_layout()
fig.savefig('data/processed/residuals_by_segment.png', dpi=110)
plt.close(fig)
print('Saved data/processed/residuals_by_segment.png')

Saved data/processed/residuals_by_segment.png


## Bootstrap a Metric

In [9]:
bm = bootstrap_metric(y, df2['y_hat'].values, mae, n_boot=600)
print(f"MAE bootstrap: mean={bm['mean']:.4f}, 95% CI=[{bm['lo']:.4f}, {bm['hi']:.4f}]")

MAE bootstrap: mean=1.2714, 95% CI=[1.0551, 1.5310]


## Stakeholder Summary

**What this model does.** A single-feature linear fit predicting `y_target` from `x_feature`,
with 5% of `x_feature` missing and filled in before fitting.

**Key assumptions.** The gaussian-based confidence band assumes normally distributed residuals.
This dataset was generated with heavy-tailed noise on purpose, and it shows: the bootstrap CI
comes out about 30% wider than the gaussian one (0.79 vs 0.61 average width) across the same
prediction line. The gaussian band is understating uncertainty here, not by a small margin.

**Sensitivity results.** Mean and median imputation land at essentially the same MAE (1.278 vs
1.284), so the imputation *method* barely matters for this dataset. Dropping the missing rows
instead lands at a noticeably lower MAE (1.065), but that comparison isn't quite apples to
apples: it's measured on 171 rows instead of 180, and the 9 imputed rows are exactly the ones
most likely to fit the line poorly, since they were filled with a single central value rather
than their real `x_feature`. Read together: the choice to impute vs drop matters more than
which imputation method is used, and a lower MAE from dropping data isn't automatically the
right choice if the missing rows are needed for other reasons.

**Subgroup risk.** Segment C's residuals are noticeably more spread out (std 2.81) than
Segment A (1.83) or Segment B (1.71), even though all three have residual means close to zero,
no group is systematically over- or under-predicted, but Segment C's predictions are less
precise. Segment C is also the smallest group (45 of 180 rows), so part of that spread could be
sample-size noise rather than a real difference, worth re-checking as more Segment C data
comes in.

**Bottom line for a stakeholder.** Prediction MAE is about 1.27, with a bootstrap 95% CI of
[1.06, 1.53], use the wider bootstrap interval, not a gaussian one, when communicating
uncertainty, since the underlying noise is heavier-tailed than gaussian. The model is
reasonably stable to the imputation method chosen, but Segment C predictions carry meaningfully
more uncertainty than Segments A or B and should be flagged as such rather than reported with
the same confidence.